<h1 style="color:#C2185B">Data Preparation</h1>

<h2 style="color:#C2185B">Owner</h2>

<h3 style="color:#C2185B">Goal</h3>

Load the raw IEEE-CIS data, clean and encode it, and save a single reusable cleaned dataset that 02 through 05 all build on.

<h4 style="color:#C2185B">Outline</h4>

*Feel free to expand on anything below, this is just a starting skeleton, not a strict limit on what you explore or add.*

- **[I. Configuration & Setup](#config-setup)**
- **[II. Data Loading and Quality Checks](#data-loading-quality-checks)**
  1. [Info](#info)
  2. [Shape](#shape)
  3. [Duplicate Values](#duplicate-values)
  4. [Missing Values](#missing-values)
  5. [Outliers and Value Range Check](#outliers-range-check)
- **[III. Cleaning & Feature Prep](#cleaning-feature-prep)**
  1. [Separate label from the rest of the data](#separate-label)
  2. [Drop ID / leakage columns](#drop-id-leakage)
  3. [Drop high-missingness columns (from II.4)](#drop-high-missingness)
  4. [Define numerical and categorical columns](#define-numerical-categorical)
  5. [Handling missing values](#handle-missing-values)
  6. [Encoding](#encoding)
- **[IV. Save cleaned data to parquet](#save-parquet)**
- **[V. Sanity Check](#sanity-check)**

<a id="config-setup"></a>
<h1 style="color:#C2185B">I. Configuration & Setup</h1>

In [8]:
# Import libraries
import os
import time
import pandas as pd 
import numpy

In [ ]:
# Configuration & Setup

# Define base directory for the project: walk up from the current working
# directory until a file that only exists at the repo root is found.
PROJECT_MARKERS = ('.git', 'requirements.txt')

def find_project_root(start_dir=None, markers=PROJECT_MARKERS):
    current = os.path.abspath(start_dir or os.getcwd())
    while True:
        if any(os.path.exists(os.path.join(current, marker)) for marker in markers):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"Could not locate project root (looked for {markers})")
        current = parent

BASE_DIR = find_project_root()

# Define the path to the data directory
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DATA_DIR = os.path.join(DATA_DIR, 'raw')
CLEANED_DATA_DIR = os.path.join(DATA_DIR, 'cleaned')
os.makedirs(CLEANED_DATA_DIR, exist_ok=True)

# Print
print(f"Project root directory: {BASE_DIR}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Cleaned data directory: {CLEANED_DATA_DIR}")

<a id="data-loading-quality-checks"></a>
<h1 style="color:#C2185B">II. Data Loading and Quality Checks</h1>

In [ ]:
# Import data from the raw data directory
# Make sure you download data from the IEEE-CIS Fraud Detection competition on Kaggle and 
# place it under data/raw (see data/README.md for the exact download steps).

# Define the paths to the transaction and identity CSV files
TRANSACTION_PATH = os.path.join(RAW_DATA_DIR, 'train_transaction.csv')
IDENTITY_PATH = os.path.join(RAW_DATA_DIR, 'train_identity.csv')

# Check if the required data files exist before trying to load them
required_paths = {
    "transaction": TRANSACTION_PATH,
    "identity": IDENTITY_PATH,
}
missing = [name for name, path in required_paths.items() if not os.path.exists(path)]

if missing:
    for name in missing:
        print(f"⚠️ Missing {name} data. Make sure to download the IEEE-CIS dataset and upload it "
              f"into data/raw first, see data/README.md for the exact steps.")
else:
    train_transaction = pd.read_csv(TRANSACTION_PATH)
    train_identity = pd.read_csv(IDENTITY_PATH)
    df = train_transaction.merge(train_identity, on='TransactionID', how='left')
    print(f"✅ Loaded and merged data: {df.shape[0]} rows, {df.shape[1]} columns.")

⚠️ Missing transaction data. Make sure to download the IEEE-CIS dataset and upload into data/raw first, see data/README.md for the exact steps.
⚠️ Missing identity data. Make sure to download the IEEE-CIS dataset and upload into data/raw first, see data/README.md for the exact steps.


<a id="info"></a>
<h2 style="color:#C2185B">1. Info</h2>

In [ ]:
# Preview the first few rows of train_transaction
# Goal: get a first look at the columns and values before doing anything else.

In [ ]:
# Preview the first few rows of train_identity
# Goal: same as above, for the identity file.

In [ ]:
# Show column names, data types, and non-null counts for train_transaction
# Example: in pandas, you can use .info()
# Goal: spot obvious type issues and missing data before cleaning.

In [ ]:
# Show column names, data types, and non-null counts for train_identity
# Example: in pandas, you can use .info()
# Goal: same as above, for the identity file.

In [ ]:
# Preview the first few rows of df (the merged dataset)
# Goal: confirm the merge looks right before relying on df going forward.

In [ ]:
# Show column names, data types, and non-null counts for df, the merged dataset
# Example: in pandas, you can use .info()
# Goal: same idea as the per-file info checks above, but on the combined data.

In [ ]:
# Target variable check
# Goal: confirm isFraud is safe to rely on before using it anywhere later.
# - data type: should be numeric (int or bool), not text
# - unique values: should be exactly 0 and 1, nothing else (e.g. no stray NaN)
# - fraud rate: the mean of the column, the overall percentage of fraud transactions

In [ ]:
# Data type summary across the merged data
# Goal: count how many columns are numeric vs. text/categorical (object data type),
# so section III.4 (Define numerical and categorical columns) has a starting point.

**Comment:**

<a id="shape"></a>
<h2 style="color:#C2185B">2. Shape</h2>

In [ ]:
# Check shapes of the source files and the merged data, and make sure it matches what you expect.
# Goal: a quick number to compare against the merge sanity check below.

In [ ]:
# Confirm the merge didn't duplicate rows: it's a left join on TransactionID
# from train_transaction, so df should have the same number of rows as
# train_transaction. If not, TransactionID isn't unique in train_identity
# and the join duplicated rows.

**Comment:**

<a id="duplicate-values"></a>
<h2 style="color:#C2185B">3. Duplicate Values</h2>

In [ ]:
# Check for fully duplicated rows in df
# Example: in pandas, you can use .duplicated().sum()
# Goal: catch accidental duplicate records before they skew the model.

In [ ]:
# Check for duplicate TransactionID values in df
# Example: in pandas, you can use df['TransactionID'].duplicated().sum()
# Goal: TransactionID should be a unique identifier, so any duplicates here
# would mean the merge in section II went wrong.

**Comment:**

<a id="missing-values"></a>
<h2 style="color:#C2185B">4. Missing Values</h2>

In [ ]:
# Build a summary table of missing values: one row per column, with the
# missing value count and percent missing, sorted from most to least missing.
# Example: in pandas, you can combine df.isnull().sum() and df.isnull().mean() * 100
# into a new DataFrame.
# Store it in a named variable, e.g. missing_summary, since section III.3 will
# reference it directly to decide which columns to drop.
# Goal: see where missingness actually lives in the data before deciding
# anything about it.

**Comment:**

<a id="outliers-range-check"></a>
<h2 style="color:#C2185B">5. Outliers and Value Range Check</h2>

In [ ]:
# Look at summary statistics for all numeric columns (min, max, mean, quartiles)
# Example: in pandas, you can use df.describe()
# Goal: spot values that look impossible or extreme before deciding how to
# handle them.

In [ ]:
# Check the range of TransactionAmt specifically
# Example: in pandas, you can look at .min(), .max(), and a histogram
# Goal: transaction amounts should be positive; flag anything that looks
# impossible (zero, negative, or an unrealistic maximum).

**Comment:**

<a id="cleaning-feature-prep"></a>
<h1 style="color:#C2185B">III. Cleaning & Feature Prep</h1>

<a id="separate-label"></a>
<h2 style="color:#C2185B">1. Separate label from the rest of the data</h2>

In [ ]:
# Confirm isFraud is set aside before any further cleaning
# Keep working on df through the rest of section III (don't fork into a
# separate features variable yet); the actual split into features/label
# files happens once, at the end, in section IV
# Goal: make sure isFraud is never touched by the encoding or missing-value
# steps below, e.g. it shouldn't appear in whatever column list III.4 builds.

**Comment:**

<a id="drop-id-leakage"></a>
<h2 style="color:#C2185B">2. Drop ID / leakage columns</h2>

In [ ]:
# Drop identifier columns that aren't real features
# Example: TransactionID is a unique identifier, not a feature; also take a
# second look at card1-card6, some of these may function as IDs rather than
# numeric magnitudes
# Goal: keep only columns that could plausibly carry a fraud signal, and
# write down why each dropped column was dropped.

**Comment:**

<a id="drop-high-missingness"></a>
<h2 style="color:#C2185B">3. Drop high-missingness columns</h2>

In [ ]:
# Use missing_summary from II.4 to decide which columns to drop
# Example: pick a missingness threshold (e.g. drop columns that are more than
# 90% missing) and drop them
# Goal: don't carry columns that are mostly empty into modeling; document the
# threshold you chose and why.

**Comment:**

<a id="define-numerical-categorical"></a>
<h2 style="color:#C2185B">4. Define numerical and categorical columns</h2>

In [ ]:
# Split the remaining columns into numerical and categorical lists
# Example: use the dtype summary from II.1 as a starting point, then check by
# hand, e.g. TransactionDT is a time offset, not a real timestamp, and some
# numeric-looking columns (like card1-card6) might actually belong with the
# categoricals instead
# Store the result as two named lists, e.g. numerical_cols and categorical_cols
# (leave isFraud out of both), since III.5 and III.6 will both reference them
# directly.
# Goal: a clear list of which columns go through encoding (III.6) vs which
# stay numeric as-is.

**Comment:**

<a id="handle-missing-values"></a>
<h2 style="color:#C2185B">5. Handling missing values</h2>

In [ ]:
# Fill in missing values for the remaining columns
# Uses numerical_cols and categorical_cols from III.4 to decide which fill
# strategy applies to which column
# Example: for numeric columns, use the median (e.g. df[col].fillna(df[col].median()));
# for categorical columns, use an explicit category like 'missing' instead of
# dropping or guessing a value
# Goal: no remaining column should have missing values after this step;
# document which strategy was used for which columns.

In [ ]:
# Sanity check: confirm no missing values remain after handling them above
remaining_missing = df.isnull().sum().sum()

if remaining_missing == 0:
    print("✅ No missing values have been found.")
else:
    print(f"⚠️ {remaining_missing} missing values still remain. Check the handling step above.")

**Comment:**

<a id="encoding"></a>
<h2 style="color:#C2185B">6. Encoding</h2>

In [ ]:
# Encode categorical columns into numeric form
# Uses categorical_cols from III.4 to know which columns need encoding
# Example: one-hot encoding (pd.get_dummies) works well for low-cardinality; 
# for high-cardinality ones like DeviceInfo or P_emaildomain/R_emaildomain,
# consider grouping rare categories into an 'other' bucket first
# Goal: every remaining column should be numeric by the end of this step,
# since Isolation Forest and LOF both need numeric input.

**Comment:**

<a id="save-parquet"></a>
<h1 style="color:#C2185B">IV. Save cleaned data to parquet</h1>

In [ ]:
# Save the cleaned features and label separately to data/cleaned/
# Split out of df here, at the very end, so every step in III works on one
# object (df) instead of juggling two.

features = df.drop(columns=['isFraud'])
label = df['isFraud']

features_path = os.path.join(CLEANED_DATA_DIR, 'features.parquet')
label_path = os.path.join(CLEANED_DATA_DIR, 'label.parquet')

features.to_parquet(features_path, index=False)
label.to_frame().to_parquet(label_path, index=False)

print(f"✅ Saved cleaned features to {features_path}")
print(f"✅ Saved label to {label_path}")

**Comment:**

<a id="sanity-check"></a>
<h1 style="color:#C2185B">V. Sanity Check</h1>

Before considering this notebook done, confirm all of the following on the final saved data:

1. Make sure there are no missing values and no duplicate values in the final version.
2. Make sure `isFraud` does not appear anywhere in `features`, only in `label`.
3. Make sure every column in `features` is numeric, encoding should be complete by this point.
4. Make sure `features` and `label` have the same number of rows, and that the row order still lines up between the two.
5. Make sure the final shapes make sense: row count should match train_transaction's row count from II.2 (no duplicates expected), and column count reflects the columns you dropped plus the new encoded columns you added.